In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up paths to match project structure
PROJECT_DIR = Path("..").resolve()
EXTRACTED_CSV = PROJECT_DIR / "features" / "extracted_4pillars.csv"  # generated audio features
JAMENDO_TSV = PROJECT_DIR / "mtg-jamendo-dataset" / "data" / "autotagging_moodtheme.tsv"

print(" Notebook environment and paths initialized successfully.")

In [ ]:
# Load the audio features 
df_features = pd.read_csv(EXTRACTED_CSV)
df_features['TRACK_ID'] = df_features['TRACK_ID'].astype(str)

# Dictionary to dynamically hold tracking metrics
cascade_counts = {}
cascade_counts['Step 0: Successfully Extracted Audio'] = len(df_features)

print(f" STEP 0: Raw Baseline Features Loaded.")
print(f" Total tracks available from extraction: {len(df_features)} tracks.")

In [ ]:
# Load genre and mood/theme annotations
import pandas as pd

def read_jamendo_tag_tsv(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        header = f.readline().rstrip("\n").split("\t")
        rows = []
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 6:
                continue
            rows.append(parts[:5] + ["\t".join(parts[5:])])
    return pd.DataFrame(rows, columns=header)

def parse_jamendo_tags(tags, prefix=None):
    if not isinstance(tags, str) or not tags:
        return []
    parts = [t.strip() for t in tags.split("\t") if t.strip()]
    if prefix:
        return [t[len(prefix):] if t.startswith(prefix) else t for t in parts]
    return parts

df_genres = read_jamendo_tag_tsv(JAMENDO_TSV.parent / 'autotagging_genre.tsv')
df_moods = read_jamendo_tag_tsv(JAMENDO_TSV.parent / 'autotagging_moodtheme.tsv')

df_genres['TRACK_ID'] = df_genres['TRACK_ID'].astype(str)
df_moods['TRACK_ID'] = df_moods['TRACK_ID'].astype(str)

df_genres['GENRES'] = df_genres['TAGS'].apply(lambda tags: parse_jamendo_tags(tags, prefix='genre---'))
df_genres['GENRE_COUNT'] = df_genres['GENRES'].apply(len)

df_moods['MOODS'] = df_moods['TAGS'].apply(lambda tags: parse_jamendo_tags(tags, prefix='mood/theme---'))
df_moods['MOOD_COUNT'] = df_moods['MOODS'].apply(len)

print(f" Loaded genre annotations: {len(df_genres)} rows")
print(f" Loaded mood/theme annotations: {len(df_moods)} rows")
print(f" Unique tracks in genres: {df_genres['TRACK_ID'].nunique()}")
print(f" Unique tracks in moods: {df_moods['TRACK_ID'].nunique()}")

In [ ]:
# STEP 1: Filter tracks with at least 1 genre
# Count the number of genres per track from the GENRES list
genre_counts = df_genres['GENRE_COUNT']
tracks_with_genre = set(df_genres.loc[genre_counts >= 1, 'TRACK_ID'])

# Filter df_features to only tracks with at least 1 genre
df_filtered_step1 = df_features[df_features['TRACK_ID'].isin(tracks_with_genre)].copy()

cascade_counts['Step 1: Has ≥1 Genre'] = len(df_filtered_step1)
remaining_step1 = len(df_filtered_step1)
removed_step1 = len(df_features) - remaining_step1

print(f" STEP 1: Filter by Genre (≥1 genre required)")
print(f" Tracks with at least 1 genre: {remaining_step1}")
print(f" ❌ Tracks removed (no genre): {removed_step1}")
print(f" Progress: {remaining_step1}/{len(df_features)} tracks remaining ({100*remaining_step1/len(df_features):.1f}%)")

In [ ]:
# STEP 2: Filter tracks with at least 2 mood/theme tags
# Count the number of moods/themes per track from the MOODS list
tracks_with_moods = set(df_moods.loc[df_moods['MOOD_COUNT'] >= 2, 'TRACK_ID'])

# Filter df_filtered_step1 to only tracks with at least 2 moods/themes
df_filtered_step2 = df_filtered_step1[df_filtered_step1['TRACK_ID'].isin(tracks_with_moods)].copy()

cascade_counts['Step 2: Has ≥2 Mood/Themes'] = len(df_filtered_step2)
remaining_step2 = len(df_filtered_step2)
removed_step2 = remaining_step1 - remaining_step2

print(f" STEP 2: Filter by Mood/Theme (≥2 moods/themes required)")
print(f" Tracks with at least 2 moods/themes: {remaining_step2}")
print(f" ❌ Tracks removed (insufficient moods/themes): {removed_step2}")
print(f" Progress: {remaining_step2}/{len(df_features)} tracks remaining ({100*remaining_step2/len(df_features):.1f}%)")
print(f"\n FILTERING SUMMARY:")
for step, count in cascade_counts.items():
 print(f" {step}: {count} tracks")

In [ ]:
# STEP 3: One-hot encode genre and mood tags
from sklearn.preprocessing import MultiLabelBinarizer

# Build a track-level dataset with raw tag lists
df_ml = df_filtered_step2[['TRACK_ID']].copy()
df_ml = df_ml.merge(
    df_genres[['TRACK_ID', 'GENRES']].drop_duplicates(subset='TRACK_ID'),
    on='TRACK_ID',
    how='left',
)
df_ml = df_ml.merge(
    df_moods[['TRACK_ID', 'MOODS']].drop_duplicates(subset='TRACK_ID'),
    on='TRACK_ID',
    how='left',
 )

# Ensure missing tag rows become empty lists
df_ml['GENRES'] = df_ml['GENRES'].apply(lambda x: x if isinstance(x, list) else [])
df_ml['MOODS'] = df_ml['MOODS'].apply(lambda x: x if isinstance(x, list) else [])

# Fit and transform one-hot encoders
mlb_genres = MultiLabelBinarizer()
mlb_moods = MultiLabelBinarizer()

df_targets_genres = pd.DataFrame(
    mlb_genres.fit_transform(df_ml['GENRES']),
    columns=[f"genre_{c}" for c in mlb_genres.classes_],
    index=df_ml.index,
 )

df_targets_moods = pd.DataFrame(
    mlb_moods.fit_transform(df_ml['MOODS']),
    columns=[f"mood_{c}" for c in mlb_moods.classes_],
    index=df_ml.index,
 )

# Combine one-hot targets with the track-level data
df_ml = pd.concat([df_ml, df_targets_genres, df_targets_moods], axis=1)

# Merge audio feature columns from df_filtered_step2 into df_ml so X and Y align
audio_feature_cols = [c for c in df_filtered_step2.columns if c != 'TRACK_ID' and c not in ('GENRES','GENRE_COUNT','MOODS','MOOD_COUNT')]
# Fallback: choose numeric columns if the above set is empty
if not audio_feature_cols:
    audio_feature_cols = [c for c in df_filtered_step2.columns if c != 'TRACK_ID' and pd.api.types.is_numeric_dtype(df_filtered_step2[c])]
df_ml = df_ml.merge(df_filtered_step2[['TRACK_ID'] + audio_feature_cols], on='TRACK_ID', how='left')

print(' One-hot encoding complete')
print(f" Tracks encoded: {len(df_ml)}")
print(f" Genre target columns: {df_targets_genres.shape[1]}")
print(f" Mood target columns: {df_targets_moods.shape[1]}")
print(f" Audio feature columns merged: {len(audio_feature_cols)}")

In [ ]:
# STEP 4: Prune sparse genres/moods, re-encode targets, and split
from sklearn.model_selection import train_test_split

MIN_OCCURRENCES = 100
print(" Step 4: Pruning rare tags and rebuilding the pruned training set..")

prune_source = df_ml[['TRACK_ID', 'GENRES', 'MOODS'] + audio_feature_cols].copy()
prune_source['GENRES'] = prune_source['GENRES'].apply(lambda x: x if isinstance(x, list) else [])
prune_source['MOODS'] = prune_source['MOODS'].apply(lambda x: x if isinstance(x, list) else [])

genre_counts = prune_source['GENRES'].explode().value_counts(dropna=True)
mood_counts = prune_source['MOODS'].explode().value_counts(dropna=True)

valid_genres = set(genre_counts[genre_counts >= MIN_OCCURRENCES].index)
valid_moods = set(mood_counts[mood_counts >= MIN_OCCURRENCES].index)

print(f" Raw genre labels: {len(genre_counts)}")
print(f" Raw mood labels: {len(mood_counts)}")
print(f" Genres with ≥{MIN_OCCURRENCES}: {len(valid_genres)}")
print(f" Moods with ≥{MIN_OCCURRENCES}: {len(valid_moods)}")

prune_source['GENRES'] = prune_source['GENRES'].apply(lambda tags: [g for g in tags if g in valid_genres])
prune_source['MOODS'] = prune_source['MOODS'].apply(lambda tags: [m for m in tags if m in valid_moods])

before_prune = len(prune_source)
pruned_df = prune_source[(prune_source['GENRES'].str.len() > 0) & (prune_source['MOODS'].str.len() > 0)].reset_index(drop=True)
after_prune = len(pruned_df)

print(f" Tracks before sparse-tag pruning: {before_prune}")
print(f" Tracks after sparse-tag pruning: {after_prune}")
print(f" Tracks removed by tag pruning: {before_prune - after_prune}")

mlb_genres_pruned = MultiLabelBinarizer()
mlb_moods_pruned = MultiLabelBinarizer()

df_pruned_genres = pd.DataFrame(
    mlb_genres_pruned.fit_transform(pruned_df['GENRES']),
    columns=[f"genre_{c}" for c in mlb_genres_pruned.classes_],
    index=pruned_df.index,
)

df_pruned_moods = pd.DataFrame(
    mlb_moods_pruned.fit_transform(pruned_df['MOODS']),
    columns=[f"mood_{c}" for c in mlb_moods_pruned.classes_],
    index=pruned_df.index,
)

pruned_genre_cols = list(df_pruned_genres.columns)
pruned_mood_cols = list(df_pruned_moods.columns)

# Rebuild the pruned master data frame with audio features and re-encoded targets.
df_pruned = pd.concat(
    [pruned_df[audio_feature_cols].reset_index(drop=True), df_pruned_genres.reset_index(drop=True), df_pruned_moods.reset_index(drop=True)],
    axis=1,
)

print(f" Pruned dataset genres: {len(pruned_genre_cols)}")
print(f" Pruned dataset moods: {len(pruned_mood_cols)}")

# Train/test split on the pruned master dataset.
X_audio_raw = df_pruned[audio_feature_cols].values
Y_mood = df_pruned[pruned_mood_cols].values
Y_genre = df_pruned[pruned_genre_cols].values

X_train_audio_raw, X_test_audio_raw, Y_train_mood, Y_test_mood, Y_train_genre, Y_test_genre = train_test_split(
    X_audio_raw, Y_mood, Y_genre, test_size=0.20, random_state=42
)

# Use XGBoost on raw audio feature space, without scaling or PCA.
X_train = X_train_audio_raw
X_test = X_test_audio_raw

print("==========================================================================")
print(" PRUNED DATASET SPLIT COMPLETE")
print("==========================================================================")
print(f" Retained genres: {len(pruned_genre_cols)}")
print(f" Retained moods: {len(pruned_mood_cols)}")
print(f" Train samples: {len(X_train)}")
print(f" Test samples: {len(X_test)}")
print("==========================================================================")

genre_target_cols = pruned_genre_cols
mood_target_cols = pruned_mood_cols


In [ ]:
# STEP 5: Train Path A (Audio -> Mood) and Path C (Audio -> Genre)
import xgboost as xgb
import numpy as np

def train_binary_relevance_ensemble(X_train, Y_train, label_names, domain_name):
    """Generic loop helper to train a Binary Relevance XGBoost array cleanly."""
    models = {}
print(f"\n Training {domain_name} Ensemble ({len(label_names)} tags)..")
    
    for idx, tag in enumerate(label_names):
        # Calculate imbalance ratio to support rare tags safely
        num_neg = np.sum(Y_train[:, idx] == 0)
        num_pos = np.sum(Y_train[:, idx] == 1)
        
        # If a tag doesn't exist in training split at all, skip to avoid a crash
        if num_pos == 0:
print(f" Skipping '{tag}' - no active positive samples in training split.")
            continue
            
        imbalance_ratio = num_neg / num_pos
        
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            scale_pos_weight=imbalance_ratio,
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1  # Uses all Mac CPU cores for lightning speed
        )
        
        # Train on this specific binary tag column
        model.fit(X_train, Y_train[:, idx])
        models[tag] = model
        
        # Keep track of progress every 15 models
        if (idx + 1) % 15 == 0 or (idx + 1) == len(label_names):
print(f" Trained [{idx + 1}/{len(label_names)}] {domain_name} models..")
            
    return models

# 1. Train Path A: Audio -> Mood
models_path_A = train_binary_relevance_ensemble(
    X_train, Y_train_mood, mood_target_cols, "Path A: Mood"
)

# 2. Train Path C: Audio -> Genre
models_path_C = train_binary_relevance_ensemble(
    X_train, Y_train_genre, genre_target_cols, "Path C: Genre"
)

print("\n Success. Path A and Path C baseline ensembles are fully trained and saved in memory")


In [ ]:
# STEP 6: Evaluate Path A (Mood) and Path C (Genre) on Test Set
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, hamming_loss
import pandas as pd

def evaluate_binary_relevance_ensemble(models, X_test, Y_test, label_names, domain_name):
    """Evaluate Binary Relevance ensemble on test set with multiple metrics."""
    
 print(f"\n Evaluating {domain_name} Ensemble on Test Set ({X_test.shape[0]} samples)..")
 print("=" * 80)
    
    # Predict on test set for each model
    Y_pred_binary = np.zeros((X_test.shape[0], len(label_names)))
    Y_pred_proba = np.zeros((X_test.shape[0], len(label_names)))
    
    for idx, tag in enumerate(label_names):
        if tag in models:
            Y_pred_binary[:, idx] = models[tag].predict(X_test)
            Y_pred_proba[:, idx] = models[tag].predict_proba(X_test)[:, 1]
        else:
            # Tag was skipped during training (no positive samples)
            Y_pred_binary[:, idx] = 0
            Y_pred_proba[:, idx] = 0.0
    
    # Calculate overall metrics
    hamming = hamming_loss(Y_test, Y_pred_binary)
    accuracy = accuracy_score(Y_test, Y_pred_binary)
    micro_f1 = f1_score(Y_test, Y_pred_binary, average='micro', zero_division=0)
    macro_f1 = f1_score(Y_test, Y_pred_binary, average='macro', zero_division=0)
    
 print(f"\n OVERALL METRICS:")
 print(f" Hamming Loss: {hamming:.4f} (lower is better)")
 print(f" Accuracy: {accuracy:.4f}")
 print(f" Micro F1-Score: {micro_f1:.4f} (pools all predictions globally)")
 print(f" Macro F1-Score: {macro_f1:.4f} (averages per-label, catches rare tags)")
    
    # Per-tag metrics
 print(f"\n PER-{domain_name.split(':')[1].strip().upper()} METRICS:")
 print(f"{'Tag':<25} {'Accuracy':>10} {'F1-Score':>10} {'AUC-ROC':>10} {'Samples':>10}")
 print("-" * 60)
    
    results_list = []
    for idx, tag in enumerate(label_names):
        if tag not in models:
            # Tag was skipped - all zeros predicted
            results_list.append({
                'Tag': tag,
                'Accuracy': 0.0,
                'F1-Score': 0.0,
                'AUC-ROC': np.nan,
                'Positive_Samples': np.sum(Y_test[:, idx])
            })
 print(f"{tag:<25} {'SKIPPED':>10} {'SKIPPED':>10} {'SKIPPED':>10} {int(np.sum(Y_test[:, idx])):>10}")
        else:
            acc = accuracy_score(Y_test[:, idx], Y_pred_binary[:, idx])
            f1 = f1_score(Y_test[:, idx], Y_pred_binary[:, idx], zero_division=0)
            
            # Only calculate AUC-ROC if we have both positive and negative samples
            pos_samples = np.sum(Y_test[:, idx] == 1)
            neg_samples = np.sum(Y_test[:, idx] == 0)
            if pos_samples > 0 and neg_samples > 0:
                auc = roc_auc_score(Y_test[:, idx], Y_pred_proba[:, idx])
            else:
                auc = np.nan
            
            results_list.append({
                'Tag': tag,
                'Accuracy': acc,
                'F1-Score': f1,
                'AUC-ROC': auc,
                'Positive_Samples': int(pos_samples)
            })
            
            auc_str = f"{auc:.4f}" if not np.isnan(auc) else "N/A"
 print(f"{tag:<25} {acc:>10.4f} {f1:>10.4f} {auc_str:>10} {int(pos_samples):>10}")
    
 print("=" * 80)
    
    return {
        'Y_pred_binary': Y_pred_binary,
        'Y_pred_proba': Y_pred_proba,
        'hamming_loss': hamming,
        'accuracy': accuracy,
        'micro_f1': micro_f1,
        'macro_f1': macro_f1,
        'results_df': pd.DataFrame(results_list)
    }

# Evaluate Path A: Mood Predictions
results_mood = evaluate_binary_relevance_ensemble(
    models_path_A, X_test, Y_test_mood, mood_target_cols, "Path A: Mood"
)

# Evaluate Path C: Genre Predictions
results_genre = evaluate_binary_relevance_ensemble(
    models_path_C, X_test, Y_test_genre, genre_target_cols, "Path C: Genre"
)


In [ ]:
# STEP 7: Feature Importance - Top Audio Features per Mood/Genre
import matplotlib.pyplot as plt

def extract_top_features_per_label(models, label_names, audio_feature_cols, domain_name, top_k=10):
    """Extract and visualize top-k most important audio features for each label."""
    
    print(f"\n🔍 Top {top_k} Audio Features for {domain_name}:")
    print("=" * 80)
    
    feature_importance_dict = {}
    
    for tag in label_names:
        if tag in models:
            model = models[tag]
            importances = model.feature_importances_
            
            # Get top-k indices
            top_indices = np.argsort(importances)[-top_k:][::-1]
            top_features = [audio_feature_cols[i] for i in top_indices]
            top_values = importances[top_indices]
            
            feature_importance_dict[tag] = {
                'features': top_features,
                'importances': top_values
            }
            
    print(f"\n {tag}:")
    for rank, (feat, imp) in enumerate(zip(top_features, top_values), 1):
         bar = "█" * int(imp * 50)  # Visual bar
    print(f" {rank:2d}. {feat:<30} {imp:.6f} {bar}")
    
    print("\n" + "=" * 80)
    
    return feature_importance_dict

# Extract top 10 features for moods
top_features_mood = extract_top_features_per_label(
    models_path_A, mood_target_cols, audio_feature_cols, "Path A: Mood", top_k=10
)

# Extract top 10 features for genres
top_features_genre = extract_top_features_per_label(
    models_path_C, genre_target_cols, audio_feature_cols, "Path C: Genre", top_k=10
)

print("\n Feature importance extraction complete")

In [ ]:
# STEP 7: Path B - Training the Mood -> Genre Bridge with classifier stacking
import xgboost as xgb
import numpy as np
from sklearn.metrics import hamming_loss, f1_score

# 1. Generate continuous mood probability features for train and test.
print(" Generating continuous mood probability features for stacking..")
Y_train_mood_prob = np.zeros(Y_train_mood.shape, dtype=float)
Y_test_mood_prob = np.zeros(Y_test_mood.shape, dtype=float)
for idx, mood_tag in enumerate(mood_target_cols):
    if mood_tag in models_path_A:
        Y_train_mood_prob[:, idx] = models_path_A[mood_tag].predict_proba(X_train)[:, 1]
        Y_test_mood_prob[:, idx] = models_path_A[mood_tag].predict_proba(X_test)[:, 1]
    else:
        Y_train_mood_prob[:, idx] = 0.0
        Y_test_mood_prob[:, idx] = 0.0

# 2. Train Path B stacking models on probabilistic mood features.
print(" Training Path B ensemble on continuous mood probabilities..")
models_path_B = {}
for idx, genre_tag in enumerate(genre_target_cols):
    pos = int(np.sum(Y_train_genre[:, idx] == 1))
    neg = int(np.sum(Y_train_genre[:, idx] == 0))
    if pos == 0:
 print(f" Skipping genre '{genre_tag}' - no positive samples in train split.")
        continue
    imbalance_ratio = float(neg) / max(1.0, float(pos))
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        scale_pos_weight=imbalance_ratio,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1,
    )
    model.fit(Y_train_mood_prob, Y_train_genre[:, idx])
    models_path_B[genre_tag] = model

# 3. Evaluate Path B using true moods and predicted mood probabilities.
Y_pred_B1 = np.zeros(Y_test_genre.shape, dtype=int)
Y_pred_B2 = np.zeros(Y_test_genre.shape, dtype=int)
for idx, genre_tag in enumerate(genre_target_cols):
    if genre_tag in models_path_B:
        Y_pred_B1[:, idx] = models_path_B[genre_tag].predict(Y_test_mood)
        Y_pred_B2[:, idx] = models_path_B[genre_tag].predict(Y_test_mood_prob)

metrics_B1 = {
    'Hamming Loss': hamming_loss(Y_test_genre, Y_pred_B1),
    'Micro F1-Score': f1_score(Y_test_genre, Y_pred_B1, average='micro', zero_division=0),
    'Macro F1-Score': f1_score(Y_test_genre, Y_pred_B1, average='macro', zero_division=0),
}
metrics_B2 = {
    'Hamming Loss': hamming_loss(Y_test_genre, Y_pred_B2),
    'Micro F1-Score': f1_score(Y_test_genre, Y_pred_B2, average='micro', zero_division=0),
    'Macro F1-Score': f1_score(Y_test_genre, Y_pred_B2, average='macro', zero_division=0),
}

# 4. Print comparison table for Path C, B1, and B2.
metrics_C_test = {
    'Hamming Loss': results_genre.get('hamming_loss', np.nan),
    'Micro F1-Score': results_genre.get('micro_f1', np.nan),
    'Macro F1-Score': results_genre.get('macro_f1', np.nan),
}

print("\n" + "="*110)
print(" FINAL PATH COMPARISON")
print("="*110)
print(f"{'Metric':<22} | {'Path C (Audio->Genre)':<24} | {'Path B1 (TrueMood->Genre)':<26} | {'Path B2 (PredMoodProb->Genre)':<28}")
print("-"*110)
for metric in ['Hamming Loss', 'Micro F1-Score', 'Macro F1-Score']:
 print(f"{metric:<22} | {metrics_C_test[metric]:<24.4f} | {metrics_B1[metric]:<26.4f} | {metrics_B2[metric]:<28.4f}")
print("="*110)


In [ ]:
# STEP 8: Classifier Chain Ensemble for Genre label dependency
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier

base_estimator = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
)

chain_ensemble = []
for i in range(5):
    chain = ClassifierChain(
        estimator=base_estimator,
        order='random',
        random_state=i,
    )
    chain.fit(X_train, Y_train_genre)
    chain_ensemble.append(chain)

prob_matrices = []
for chain in chain_ensemble:
    chain_proba = chain.predict_proba(X_test)
    if isinstance(chain_proba, list):
        chain_proba = np.vstack([p[:, 1] for p in chain_proba]).T
    elif hasattr(chain_proba, 'ndim') and chain_proba.ndim == 3:
        chain_proba = chain_proba[:, :, 1]
    prob_matrices.append(chain_proba)

Y_pred_prob_ensemble = np.mean(prob_matrices, axis=0)
Y_pred_bin_ensemble = (Y_pred_prob_ensemble >= 0.5).astype(int)

cc_hamming = hamming_loss(Y_test_genre, Y_pred_bin_ensemble)
cc_micro = f1_score(Y_test_genre, Y_pred_bin_ensemble, average='micro', zero_division=0)
cc_macro = f1_score(Y_test_genre, Y_pred_bin_ensemble, average='macro', zero_division=0)

baseline_path_c = {
    'Hamming Loss': 0.0873,
    'Micro F1-Score': 0.4219,
    'Macro F1-Score': 0.2873,
}

print("\n" + "="*90)
print(" CLASSIFIER CHAIN ENSEMBLE RESULTS")
print("="*90)
print(f"Ensemble size: {len(chain_ensemble)} chains")
print(f"Test samples: {X_test.shape[0]}")
print("\n")
print(f"{'Metric':<22} | {'Classifier Chain Ensemble':<28} | {'Path C Baseline':<18}")
print("-"*90)
print(f"{'Hamming Loss':<22} | {cc_hamming:<28.4f} | {baseline_path_c['Hamming Loss']:<18.4f}")
print(f"{'Micro F1-Score':<22} | {cc_micro:<28.4f} | {baseline_path_c['Micro F1-Score']:<18.4f}")
print(f"{'Macro F1-Score':<22} | {cc_macro:<28.4f} | {baseline_path_c['Macro F1-Score']:<18.4f}")
print("="*90)

In [ ]:
# STEP 9: Paired bootstrap significance test for Path C vs Path B2 on the global test set
from sklearn.metrics import f1_score

# Isolate the required matrices from previously computed results
Y_pred_genre_C = results_genre['Y_pred_binary']
Y_pred_genre_B2 = Y_pred_B2

assert Y_test_genre.shape == Y_pred_genre_C.shape == Y_pred_genre_B2.shape, (
    "Shape mismatch: Y_test_genre, Y_pred_genre_C, and Y_pred_genre_B2 must match"
)


def paired_bootstrap_f1(Y_true, Y_pred_A, Y_pred_B, average, n_iterations=2000):
    boot_A = []
    boot_B = []
    boot_delta = []
    n_samples = Y_true.shape[0]

    for i in range(n_iterations):
        rng = np.random.default_rng(seed=i)
        indices = rng.integers(0, n_samples, size=n_samples)

        Y_true_bs = Y_true[indices]
        Y_A_bs = Y_pred_A[indices]
        Y_B_bs = Y_pred_B[indices]

        f1_A = f1_score(Y_true_bs, Y_A_bs, average=average, zero_division=0)
        f1_B = f1_score(Y_true_bs, Y_B_bs, average=average, zero_division=0)

        boot_A.append(f1_A)
        boot_B.append(f1_B)
        boot_delta.append(f1_A - f1_B)

    boot_A = np.array(boot_A)
    boot_B = np.array(boot_B)
    boot_delta = np.array(boot_delta)

    ci_low, ci_high = np.percentile(boot_delta, [2.5, 97.5])
    p_val = 2.0 * min(np.mean(boot_delta <= 0.0), np.mean(boot_delta >= 0.0))
    p_val = min(p_val, 1.0)

    observed_A = f1_score(Y_true, Y_pred_A, average=average, zero_division=0)
    observed_B = f1_score(Y_true, Y_pred_B, average=average, zero_division=0)
    observed_delta = observed_A - observed_B

    return {
        'observed_A': observed_A,
        'observed_B': observed_B,
        'observed_delta': observed_delta,
        'boot_mean_A': boot_A.mean(),
        'boot_mean_B': boot_B.mean(),
        'boot_delta': boot_delta,
        'ci': (ci_low, ci_high),
        'p_value': p_val,
        'n_iterations': n_iterations,
    }


# Run the paired bootstrap for Macro and Micro F1
macro_results = paired_bootstrap_f1(
    Y_test_genre,
    Y_pred_genre_C,
    Y_pred_genre_B2,
    average='macro',
    n_iterations=2000,
)

micro_results = paired_bootstrap_f1(
    Y_test_genre,
    Y_pred_genre_C,
    Y_pred_genre_B2,
    average='micro',
    n_iterations=2000,
)


# Formal console summary
print("\n" + "=" * 110)
print(" PAIRED BOOTSTRAP SIGNIFICANCE TEST: PATH C vs PATH B2")
print("=" * 110)

for label, results in [('Macro F1', macro_results), ('Micro F1', micro_results)]:
    ci_low, ci_high = results['ci']
    significance = 'NOT significant' if ci_low <= 0.0 <= ci_high else 'significant'

 print(f"\n{label} RESULTS")
 print("-" * 110)
 print(f"Observed Path C {label}: {results['observed_A']:.4f}")
 print(f"Observed Path B2 {label}: {results['observed_B']:.4f}")
 print(f"Observed Δ{label}: {results['observed_delta']:.4f}")
 print(f"Bootstrap mean Path C {label}: {results['boot_mean_A']:.4f}")
 print(f"Bootstrap mean Path B2 {label}: {results['boot_mean_B']:.4f}")
 print(f"95% CI for Δ{label}: [{ci_low:.4f}, {ci_high:.4f}]")
 print(f"Two-tailed bootstrap p-value: {results['p_value']:.4f}")
 print(f"Interpretation: {significance} at α=0.05")

print("\n" + "=" * 110)
print(" Bootstrap resampling complete. This provides empirical 95% confidence intervals and paired significance assessments for the test set.")


In [ ]:
# STEP 10: Randomized hyperparameter search for Path A, Path C, and Path B
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score
import xgboost as xgb

binary_f1_scorer = make_scorer(f1_score, average='binary', zero_division=0)

param_dist = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'n_estimators': [50, 100, 150],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.5, 1.0],
}


def tune_binary_label_model(X, y, imbalance_ratio, n_iter=16, cv=3, random_state=42):
    estimator = xgb.XGBClassifier(
        objective='binary:logistic',
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=random_state,
        n_jobs=-1,
    )

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions={**param_dist, 'scale_pos_weight': [imbalance_ratio]},
        n_iter=n_iter,
        scoring=binary_f1_scorer,
        cv=cv,
        verbose=0,
        random_state=random_state,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X, y)
    return search.best_estimator_, search.best_score_, search.best_params_


def tune_binary_relevance_ensemble(X_train, Y_train, label_names, domain_name, n_iter=16, cv=3):
    tuned_models = {}
    tune_records = []

 print(f"\n🔎 Random search tuning for {domain_name} ({len(label_names)} labels)..")

    for idx, label in enumerate(label_names):
        y_label = Y_train[:, idx]
        pos = int(np.sum(y_label == 1))
        neg = int(np.sum(y_label == 0))

        if pos == 0:
 print(f" Skipping '{label}' - no positive samples in training split.")
            tune_records.append((label, None, None, None, pos, neg))
            continue

        imbalance_ratio = float(neg) / max(1.0, float(pos))
        best_model, best_score, best_params = tune_binary_label_model(
            X_train, y_label, imbalance_ratio, n_iter=n_iter, cv=cv
        )
        tuned_models[label] = best_model
        tune_records.append((label, best_score, best_params, imbalance_ratio, pos, neg))

        if (idx + 1) % 10 == 0 or (idx + 1) == len(label_names):
 print(f" Tuned [{idx + 1}/{len(label_names)}] labels for {domain_name}")

 print(f" Randomized search complete for {domain_name}.\n")
    return tuned_models, pd.DataFrame(
        tune_records,
        columns=['Label', 'Best_F1', 'Best_Params', 'Scale_Pos_Weight', 'Positives', 'Negatives'],
    )


def evaluate_models_and_compare(models, X_test, Y_test, label_names, domain_name):
    results = evaluate_binary_relevance_ensemble(models, X_test, Y_test, label_names, domain_name)
    return results

# Baseline metrics stored earlier
baseline_genre_metrics = {
    'Hamming Loss': results_genre['hamming_loss'],
    'Micro F1-Score': results_genre['micro_f1'],
    'Macro F1-Score': results_genre['macro_f1'],
}

baseline_mood_metrics = {
    'Hamming Loss': results_mood['hamming_loss'],
    'Micro F1-Score': results_mood['micro_f1'],
    'Macro F1-Score': results_mood['macro_f1'],
}

baseline_B2_metrics = {
    'Hamming Loss': metrics_B2['Hamming Loss'],
    'Micro F1-Score': metrics_B2['Micro F1-Score'],
    'Macro F1-Score': metrics_B2['Macro F1-Score'],
}

# Train tuned Path A and Path C models
models_tuned_path_A, tune_report_mood = tune_binary_relevance_ensemble(
    X_train, Y_train_mood, mood_target_cols, 'Path A: Mood', n_iter=16, cv=3
)
models_tuned_path_C, tune_report_genre = tune_binary_relevance_ensemble(
    X_train, Y_train_genre, genre_target_cols, 'Path C: Genre', n_iter=16, cv=3
)

# Generate mood probability features from tuned Path A
Y_train_mood_prob_tuned = np.zeros_like(Y_train_mood, dtype=float)
Y_test_mood_prob_tuned = np.zeros_like(Y_test_mood, dtype=float)
for idx, mood_tag in enumerate(mood_target_cols):
    if mood_tag in models_tuned_path_A:
        Y_train_mood_prob_tuned[:, idx] = models_tuned_path_A[mood_tag].predict_proba(X_train)[:, 1]
        Y_test_mood_prob_tuned[:, idx] = models_tuned_path_A[mood_tag].predict_proba(X_test)[:, 1]
    else:
        Y_train_mood_prob_tuned[:, idx] = 0.0
        Y_test_mood_prob_tuned[:, idx] = 0.0

# Train tuned Path B using tuned mood probabilities as inputs
models_tuned_path_B, tune_report_B = tune_binary_relevance_ensemble(
    Y_train_mood_prob_tuned, Y_train_genre, genre_target_cols, 'Path B: Mood->Genre', n_iter=16, cv=3
)

# Evaluate tuned models
results_mood_tuned = evaluate_models_and_compare(
    models_tuned_path_A, X_test, Y_test_mood, mood_target_cols, 'Tuned Path A: Mood'
)
results_genre_tuned = evaluate_models_and_compare(
    models_tuned_path_C, X_test, Y_test_genre, genre_target_cols, 'Tuned Path C: Genre'
)

# Evaluate tuned Path B2 (predicted mood probabilities into genre models)
Y_pred_B2_tuned = np.zeros(Y_test_genre.shape, dtype=int)
for idx, genre_tag in enumerate(genre_target_cols):
    if genre_tag in models_tuned_path_B:
        Y_pred_B2_tuned[:, idx] = models_tuned_path_B[genre_tag].predict(Y_test_mood_prob_tuned)
    else:
        Y_pred_B2_tuned[:, idx] = 0

metrics_B2_tuned = {
    'Hamming Loss': hamming_loss(Y_test_genre, Y_pred_B2_tuned),
    'Micro F1-Score': f1_score(Y_test_genre, Y_pred_B2_tuned, average='micro', zero_division=0),
    'Macro F1-Score': f1_score(Y_test_genre, Y_pred_B2_tuned, average='macro', zero_division=0),
}

print('\n' + '=' * 110)
print(' RANDOM SEARCH COMPARISON RESULTS')
print('=' * 110)
print(f"{'Path':<30} | {'Baseline Macro F1':<18} | {'Tuned Macro F1':<18} | {'Baseline Micro F1':<18} | {'Tuned Micro F1':<18}")
print('-' * 110)
print(f"{'Path A: Mood':<30} | {baseline_mood_metrics['Macro F1-Score']:<18.4f} | {results_mood_tuned['macro_f1']:<18.4f} | {baseline_mood_metrics['Micro F1-Score']:<18.4f} | {results_mood_tuned['micro_f1']:<18.4f}")
print(f"{'Path C: Genre':<30} | {baseline_genre_metrics['Macro F1-Score']:<18.4f} | {results_genre_tuned['macro_f1']:<18.4f} | {baseline_genre_metrics['Micro F1-Score']:<18.4f} | {results_genre_tuned['micro_f1']:<18.4f}")
print(f"{'Path B2: PredMoodProb->Genre':<30} | {baseline_B2_metrics['Macro F1-Score']:<18.4f} | {metrics_B2_tuned['Macro F1-Score']:<18.4f} | {baseline_B2_metrics['Micro F1-Score']:<18.4f} | {metrics_B2_tuned['Micro F1-Score']:<18.4f}")
print('=' * 110)

print('\n Randomized search completed. Tune reports are available as tune_report_mood, tune_report_genre, and tune_report_B.')

# STEP 11: Compare baseline vs tuned performance for Path C and Path B2



In [ ]:
def compare_baseline_tuned(name, baseline_metrics, tuned_metrics):
    delta_macro = tuned_metrics['macro_f1'] - baseline_metrics['Macro F1-Score']
    delta_micro = tuned_metrics['micro_f1'] - baseline_metrics['Micro F1-Score']
    better_macro = 'TUNED' if delta_macro > 0 else 'BASELINE' if delta_macro < 0 else 'EQUAL'
    better_micro = 'TUNED' if delta_micro > 0 else 'BASELINE' if delta_micro < 0 else 'EQUAL'
    return {
        'Path': name,
        'Baseline Macro F1': baseline_metrics['Macro F1-Score'],
        'Tuned Macro F1': tuned_metrics['macro_f1'],
        'Delta Macro F1': delta_macro,
        'Better Macro': better_macro,
        'Baseline Micro F1': baseline_metrics['Micro F1-Score'],
        'Tuned Micro F1': tuned_metrics['micro_f1'],
        'Delta Micro F1': delta_micro,
        'Better Micro': better_micro,
    }

comparison_rows = []
comparison_rows.append(compare_baseline_tuned(
    'Path C: Genre',
    baseline_genre_metrics,
    results_genre_tuned,
))
comparison_rows.append(compare_baseline_tuned(
    'Path B2: PredMoodProb->Genre',
    baseline_B2_metrics,
    {'macro_f1': metrics_B2_tuned['Macro F1-Score'], 'micro_f1': metrics_B2_tuned['Micro F1-Score']},
))

comparison_df = pd.DataFrame(comparison_rows)[[
    'Path',
    'Baseline Macro F1',
    'Tuned Macro F1',
    'Delta Macro F1',
    'Better Macro',
    'Baseline Micro F1',
    'Tuned Micro F1',
    'Delta Micro F1',
    'Better Micro',
]]

print('\n' + '=' * 130)
print(' Baseline vs tuned model comparison: Path C and Path B2')
print('=' * 130)
print(comparison_df.to_string(index=False, float_format='%.4f'))
print('=' * 130)

for row in comparison_rows:
 print(f"\n{row['Path']}: ")
 print(f" Macro F1: baseline={row['Baseline Macro F1']:.4f}, tuned={row['Tuned Macro F1']:.4f}, delta={row['Delta Macro F1']:+.4f} -> {row['Better Macro']}")
 print(f" Micro F1: baseline={row['Baseline Micro F1']:.4f}, tuned={row['Tuned Micro F1']:.4f}, delta={row['Delta Micro F1']:+.4f} -> {row['Better Micro']}")


In [ ]:
# STEP 14: Tuned Path C and tuned Path B2 evaluation with paired bootstrap testing
from sklearn.metrics import hamming_loss, f1_score

print(" Tuned Path C and Tuned Path B2 evaluation")
print("=" * 100)

tuned_eval_metrics = {
    'Path C Tuned (Audio -> Genre)': {
        'Hamming Loss': results_genre_tuned['hamming_loss'],
        'Micro F1-Score': results_genre_tuned['micro_f1'],
        'Macro F1-Score': results_genre_tuned['macro_f1'],
    },
    'Path B2 Tuned (PredMoodProb -> Genre)': {
        'Hamming Loss': metrics_B2_tuned['Hamming Loss'],
        'Micro F1-Score': metrics_B2_tuned['Micro F1-Score'],
        'Macro F1-Score': metrics_B2_tuned['Macro F1-Score'],
    },
}

for path_name, metrics in tuned_eval_metrics.items():
 print(f"\n{path_name}")
 print("-" * 100)
 print(f" Hamming Loss : {metrics['Hamming Loss']:.4f}")
 print(f" Micro F1-Score: {metrics['Micro F1-Score']:.4f}")
 print(f" Macro F1-Score: {metrics['Macro F1-Score']:.4f}")

Y_pred_genre_C_tuned = results_genre_tuned['Y_pred_binary']
Y_pred_genre_B2_tuned = Y_pred_B2_tuned

assert Y_test_genre.shape == Y_pred_genre_C_tuned.shape == Y_pred_genre_B2_tuned.shape, (
    'Shape mismatch: Y_test_genre, Y_pred_genre_C_tuned, and Y_pred_genre_B2_tuned must match'
)

print("\n" + "=" * 110)
print(" Paired bootstrap significance test: Tuned Path C vs Tuned Path B2")
print("=" * 110)

for label, average in [('Macro F1', 'macro'), ('Micro F1', 'micro')]:
    tuned_results = paired_bootstrap_f1(
        Y_test_genre,
        Y_pred_genre_C_tuned,
        Y_pred_genre_B2_tuned,
        average=average,
        n_iterations=2000,
    )
    ci_low, ci_high = tuned_results['ci']
    significance = 'significant' if not (ci_low <= 0.0 <= ci_high) else 'NOT significant'

 print(f"\n{label} RESULTS")
 print("-" * 110)
 print(f"Observed Tuned Path C {label}: {tuned_results['observed_A']:.4f}")
 print(f"Observed Tuned Path B2 {label}: {tuned_results['observed_B']:.4f}")
 print(f"Observed Δ{label}: {tuned_results['observed_delta']:.4f}")
 print(f"95% CI for Δ{label}: [{ci_low:.4f}, {ci_high:.4f}]")
 print(f"Two-tailed bootstrap p-value: {tuned_results['p_value']:.4f}")
 print(f"Interpretation: {significance} at α=0.05")

print("=" * 110)


In [ ]:
# STEP 15: Tag Distribution Visualizations from raw_30s_cleantags_50artists Statistics
# Load tag frequency data from pre-computed statistics

STATS_DIR = PROJECT_DIR / "mtg-jamendo-dataset" / "stats" / "raw_30s_cleantags_50artists"

# Load genre and mood/theme tag statistics
df_genre_stats = pd.read_csv(STATS_DIR / "genre.tsv", sep="\t")
df_mood_stats = pd.read_csv(STATS_DIR / "mood_theme.tsv", sep="\t")

# Sort by tracks (appearance count) in descending order
df_genre_stats = df_genre_stats.sort_values("tracks", ascending=True)
df_mood_stats = df_mood_stats.sort_values("tracks", ascending=True)

# Create figure with two subplots (taller for many genres)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 24))

# --- Genre Bar Chart ---
colors_genre = plt.cm.viridis(np.linspace(0.3, 0.9, len(df_genre_stats)))
bars1 = ax1.barh(df_genre_stats["tag"], df_genre_stats["tracks"], color=colors_genre, edgecolor="black", linewidth=0.7)
ax1.set_xlabel("Number of Track Appearances", fontsize=11, fontweight="bold")
ax1.set_ylabel("Genre Tag", fontsize=11, fontweight="bold")
ax1.set_title("Genre Tag Distribution\n(raw_30s_cleantags_50artists)", fontsize=13, fontweight="bold", pad=20)
ax1.grid(axis="x", alpha=0.3, linestyle="--")
ax1.tick_params(axis="y", labelsize=7)

# Add value labels on bars
for i, (tag, count) in enumerate(zip(df_genre_stats["tag"], df_genre_stats["tracks"])):
    ax1.text(count + 100, i, str(int(count)), va="center", fontsize=7, fontweight="bold")

# --- Mood/Theme Bar Chart ---
colors_mood = plt.cm.plasma(np.linspace(0.3, 0.9, len(df_mood_stats)))
bars2 = ax2.barh(df_mood_stats["tag"], df_mood_stats["tracks"], color=colors_mood, edgecolor="black", linewidth=0.7)
ax2.set_xlabel("Number of Track Appearances", fontsize=12, fontweight="bold")
ax2.set_ylabel("Mood/Theme Tag", fontsize=12, fontweight="bold")
ax2.set_title("Mood/Theme Tag Distribution\n(raw_30s_cleantags_50artists)", fontsize=14, fontweight="bold", pad=20)
ax2.grid(axis="x", alpha=0.3, linestyle="--")

# Add value labels on bars
for i, (tag, count) in enumerate(zip(df_mood_stats["tag"], df_mood_stats["tracks"])):
    ax2.text(count + 30, i, str(int(count)), va="center", fontsize=9, fontweight="bold")

# Overall title
fig.suptitle("Audio Tag Frequency Analysis: 50-Artist Clean Tags (30s clips)", 
             fontsize=16, fontweight="bold", y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# Print summary statistics
print("\n" + "=" * 80)
print(" TAG DISTRIBUTION SUMMARY")
print("=" * 80)
print(f"\n GENRE TAGS ({len(df_genre_stats)} total)")
print("-" * 80)
print(f"Total track appearances: {df_genre_stats['tracks'].sum():,}")
print(f"Most common: {df_genre_stats.iloc[-1]['tag']} ({int(df_genre_stats.iloc[-1]['tracks'])} appearances)")
print(f"Least common: {df_genre_stats.iloc[0]['tag']} ({int(df_genre_stats.iloc[0]['tracks'])} appearances)")
print(f"Average appearances per genre: {df_genre_stats['tracks'].mean():.1f}")

print(f"\n MOOD/THEME TAGS ({len(df_mood_stats)} total)")
print("-" * 80)
print(f"Total track appearances: {df_mood_stats['tracks'].sum():,}")
print(f"Most common: {df_mood_stats.iloc[-1]['tag']} ({int(df_mood_stats.iloc[-1]['tracks'])} appearances)")
print(f"Least common: {df_mood_stats.iloc[0]['tag']} ({int(df_mood_stats.iloc[0]['tracks'])} appearances)")
print(f"Average appearances per mood/theme: {df_mood_stats['tracks'].mean():.1f}")

print("\n Tag distribution visualization complete")
print("=" * 80)

In [ ]:
# STEP 16: Tuned Models Performance Comparison Visualization
# Create a side-by-side grouped bar chart for Path A and Path B metrics

import matplotlib.pyplot as plt
import numpy as np

# Prepare data for visualization
metrics_comparison = {
    'Path A (Audio→Genre)': {
        'Macro F1': results_genre_tuned['macro_f1'],
        'Micro F1': results_genre_tuned['micro_f1'],
        'Hamming Loss': results_genre_tuned['hamming_loss'],
    },
    'Path B (Mood→Genre)': {
        'Macro F1': metrics_B2_tuned['Macro F1-Score'],
        'Micro F1': metrics_B2_tuned['Micro F1-Score'],
        'Hamming Loss': metrics_B2_tuned['Hamming Loss'],
    }
}

# Extract paths and metrics
paths = list(metrics_comparison.keys())
metric_types = ['Macro F1', 'Micro F1', 'Hamming Loss']
x = np.arange(len(paths))
width = 0.25

# Prepare data for each metric
macro_f1_vals = [metrics_comparison[path]['Macro F1'] for path in paths]
micro_f1_vals = [metrics_comparison[path]['Micro F1'] for path in paths]
hamming_vals = [metrics_comparison[path]['Hamming Loss'] for path in paths]

# Create figure and axis
fig, ax = plt.subplots(figsize=(12, 7))

# Define colors matching the professional look
color_macro = '#2E86AB'  # Deep blue
color_micro = '#A23B72'  # Purple/magenta
color_hamming = '#F18F01'  # Orange

# Create bars
bars1 = ax.bar(x - width, macro_f1_vals, width, label='Macro F1-Score', color=color_macro, edgecolor='black', linewidth=1.2)
bars2 = ax.bar(x, micro_f1_vals, width, label='Micro F1-Score', color=color_micro, edgecolor='black', linewidth=1.2)
bars3 = ax.bar(x + width, hamming_vals, width, label='Hamming Loss', color=color_hamming, edgecolor='black', linewidth=1.2)

# Add value labels on bars
def add_value_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

add_value_labels(bars1)
add_value_labels(bars2)
add_value_labels(bars3)

# Customize the plot
ax.set_xlabel('Prediction Path', fontsize=13, fontweight='bold')
ax.set_ylabel('Score Value', fontsize=13, fontweight='bold')
ax.set_title('Tuned Models Performance Comparison\nPath A (Audio→Genre) vs Path B (Predicted Mood→Genre)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(paths, fontsize=12, fontweight='bold')
ax.legend(fontsize=11, loc='upper right', framealpha=0.95)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, max(max(macro_f1_vals), max(micro_f1_vals), max(hamming_vals)) * 1.15)

# Add subtle background
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('white')

plt.tight_layout()
plt.show()

# Print detailed comparison
print("\n" + "=" * 90)
print(" Tuned models performance metrics comparison")
print("=" * 90)
print(f"\n{'Metric':<20} | {'Path A (Audio→Genre)':<25} | {'Path B (Mood→Genre)':<25} | {'Difference':<15}")
print("-" * 90)

for metric_type in metric_types:
    path_a_val = metrics_comparison['Path A (Audio→Genre)'][metric_type]
    path_b_val = metrics_comparison['Path B (Mood→Genre)'][metric_type]
    diff = path_a_val - path_b_val
    diff_symbol = "↑" if diff > 0 else "↓" if diff < 0 else "→"
    
 print(f"{metric_type:<20} | {path_a_val:<25.4f} | {path_b_val:<25.4f} | {diff_symbol} {abs(diff):>12.4f}")

print("=" * 90)

# Summary insights
print("\n Key insights:")
print("-" * 90)
macro_delta = metrics_comparison['Path A (Audio→Genre)']['Macro F1'] - metrics_comparison['Path B (Mood→Genre)']['Macro F1']
micro_delta = metrics_comparison['Path A (Audio→Genre)']['Micro F1'] - metrics_comparison['Path B (Mood→Genre)']['Micro F1']
hamming_delta = metrics_comparison['Path A (Audio→Genre)']['Hamming Loss'] - metrics_comparison['Path B (Mood→Genre)']['Hamming Loss']

if macro_delta > 0:
 print(f" • Path A has {abs(macro_delta):.4f} higher Macro F1 (better per-label balance)")
else:
 print(f" • Path B has {abs(macro_delta):.4f} higher Macro F1 (better per-label balance)")

if micro_delta > 0:
 print(f" • Path A has {abs(micro_delta):.4f} higher Micro F1 (better global performance)")
else:
 print(f" • Path B has {abs(micro_delta):.4f} higher Micro F1 (better global performance)")

if hamming_delta > 0:
 print(f" • Path B has {abs(hamming_delta):.4f} lower Hamming Loss (fewer incorrect labels)")
else:
 print(f" • Path A has {abs(hamming_delta):.4f} lower Hamming Loss (fewer incorrect labels)")

print("=" * 90)

In [ ]:
# STEP 17: Bootstrap Distribution Histograms with Density Plots
# Visualize the distribution of Macro F1 and Micro F1 deltas from 2,000 bootstrap iterations

from scipy import stats
import matplotlib.pyplot as plt

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- Macro F1 Delta Distribution ---
macro_deltas = macro_results['boot_delta']
macro_ci_low, macro_ci_high = macro_results['ci']
macro_observed = macro_results['observed_delta']

# Plot histogram with density
ax1.hist(macro_deltas, bins=40, density=True, alpha=0.6, color='#2E86AB', edgecolor='black', linewidth=0.7, label='Bootstrap Distribution')

# Add KDE density curve
density_x = np.linspace(macro_deltas.min(), macro_deltas.max(), 200)
density_y = stats.gaussian_kde(macro_deltas)(density_x)
ax1.plot(density_x, density_y, color='#1a4d7a', linewidth=2.5, label='Kernel Density Estimate')

# Add vertical lines for CI and observed delta
ax1.axvline(macro_ci_low, color='#F18F01', linestyle='--', linewidth=2.2, label=f'95% CI Lower: {macro_ci_low:.4f}')
ax1.axvline(macro_ci_high, color='#A23B72', linestyle='--', linewidth=2.2, label=f'95% CI Upper: {macro_ci_high:.4f}')
ax1.axvline(macro_observed, color='#06A77D', linestyle='-', linewidth=2.5, label=f'Observed Δ: {macro_observed:.4f}')

# Add zero reference line
ax1.axvline(0, color='black', linestyle=':', linewidth=1.5, alpha=0.5, label='No Difference')

# Customize Macro F1 subplot
ax1.set_xlabel('Macro F1 Δ (Path A - Path B)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Density', fontsize=12, fontweight='bold')
ax1.set_title('Bootstrap Distribution: Macro F1 Δ\n(2,000 iterations)', fontsize=13, fontweight='bold', pad=15)
ax1.legend(fontsize=9, loc='upper right', framealpha=0.95)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.set_facecolor('#F8F9FA')

# --- Micro F1 Delta Distribution ---
micro_deltas = micro_results['boot_delta']
micro_ci_low, micro_ci_high = micro_results['ci']
micro_observed = micro_results['observed_delta']

# Plot histogram with density
ax2.hist(micro_deltas, bins=40, density=True, alpha=0.6, color='#A23B72', edgecolor='black', linewidth=0.7, label='Bootstrap Distribution')

# Add KDE density curve
density_x_micro = np.linspace(micro_deltas.min(), micro_deltas.max(), 200)
density_y_micro = stats.gaussian_kde(micro_deltas)(density_x_micro)
ax2.plot(density_x_micro, density_y_micro, color='#7a1a4d', linewidth=2.5, label='Kernel Density Estimate')

# Add vertical lines for CI and observed delta
ax2.axvline(micro_ci_low, color='#F18F01', linestyle='--', linewidth=2.2, label=f'95% CI Lower: {micro_ci_low:.4f}')
ax2.axvline(micro_ci_high, color='#2E86AB', linestyle='--', linewidth=2.2, label=f'95% CI Upper: {micro_ci_high:.4f}')
ax2.axvline(micro_observed, color='#06A77D', linestyle='-', linewidth=2.5, label=f'Observed Δ: {micro_observed:.4f}')

# Add zero reference line
ax2.axvline(0, color='black', linestyle=':', linewidth=1.5, alpha=0.5, label='No Difference')

# Customize Micro F1 subplot
ax2.set_xlabel('Micro F1 Δ (Path A - Path B)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Density', fontsize=12, fontweight='bold')
ax2.set_title('Bootstrap Distribution: Micro F1 Δ\n(2,000 iterations)', fontsize=13, fontweight='bold', pad=15)
ax2.legend(fontsize=9, loc='upper right', framealpha=0.95)
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.set_facecolor('#F8F9FA')

# Overall title
fig.suptitle('Paired Bootstrap Significance Testing: Path A vs Path B\nDelta Distributions with 95% Confidence Intervals', 
             fontsize=14, fontweight='bold', y=0.98)

fig.patch.set_facecolor('white')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# Print detailed bootstrap statistics
print("\n" + "=" * 100)
print(" Bootstrap resampling statistics (2,000 iterations)")
print("=" * 100)

print(f"\n{'Metric':<20} | {'Observed Δ':<15} | {'95% CI Lower':<15} | {'95% CI Upper':<15} | {'Contains Zero':<15}")
print("-" * 100)

macro_contains_zero = "YES ⚠️" if macro_ci_low <= 0 <= macro_ci_high else "NO ✓"
micro_contains_zero = "YES ⚠️" if micro_ci_low <= 0 <= micro_ci_high else "NO ✓"

print(f"{'Macro F1 Δ':<20} | {macro_observed:<15.4f} | {macro_ci_low:<15.4f} | {macro_ci_high:<15.4f} | {macro_contains_zero:<15}")
print(f"{'Micro F1 Δ':<20} | {micro_observed:<15.4f} | {micro_ci_low:<15.4f} | {micro_ci_high:<15.4f} | {micro_contains_zero:<15}")

print("=" * 100)

# Additional statistics
print("\n BOOTSTRAP DISTRIBUTION STATISTICS:")
print("-" * 100)

print(f"\nMacro F1 Δ:")
print(f" Mean of bootstrap deltas: {macro_results['boot_mean_A'] - macro_results['boot_mean_B']:.4f}")
print(f" Std dev of deltas: {np.std(macro_deltas):.4f}")
print(f" Min delta: {macro_deltas.min():.4f}")
print(f" Max delta: {macro_deltas.max():.4f}")
print(f" p-value (two-tailed): {macro_results['p_value']:.4f}")

print(f"\nMicro F1 Δ:")
print(f" Mean of bootstrap deltas: {micro_results['boot_mean_A'] - micro_results['boot_mean_B']:.4f}")
print(f" Std dev of deltas: {np.std(micro_deltas):.4f}")
print(f" Min delta: {micro_deltas.min():.4f}")
print(f" Max delta: {micro_deltas.max():.4f}")
print(f" p-value (two-tailed): {micro_results['p_value']:.4f}")

print("\n" + "=" * 100)
print(" Interpretation:")
print("-" * 100)

if macro_ci_low > 0:
 print(f" Macro F1: Path A significantly better (CI does not contain zero)")
elif macro_ci_high < 0:
 print(f" Macro F1: Path B significantly better (CI does not contain zero)")
else:
 print(f" Macro F1: No significant difference at α=0.05 (CI contains zero)")

if micro_ci_low > 0:
 print(f" Micro F1: Path A significantly better (CI does not contain zero)")
elif micro_ci_high < 0:
 print(f" Micro F1: Path B significantly better (CI does not contain zero)")
else:
 print(f" Micro F1: No significant difference at α=0.05 (CI contains zero)")

print("=" * 100)

In [ ]:
# STEP 18: Performance vs Support Scatter Plot
# Compare Path A and Path B genre-level F1 scores against support (number of samples)

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score

# Extract per-genre metrics for Path A (Audio -> Genre)
path_a_results_df = results_genre_tuned['results_df'].copy()
path_a_results_df = path_a_results_df[path_a_results_df['F1-Score'].notna()].copy()

# Generate per-genre metrics for Path B (Predicted Mood -> Genre)
path_b_genre_f1_scores = []
for idx, genre_tag in enumerate(genre_target_cols):
    if genre_tag in models_tuned_path_B:
        f1 = f1_score(Y_test_genre[:, idx], Y_pred_B2_tuned[:, idx], zero_division=0)
        pos_samples = int(np.sum(Y_test_genre[:, idx]))
        path_b_genre_f1_scores.append({
            'Tag': genre_tag,
            'F1-Score': f1,
            'Positive_Samples': pos_samples
        })
    else:
        pos_samples = int(np.sum(Y_test_genre[:, idx]))
        path_b_genre_f1_scores.append({
            'Tag': genre_tag,
            'F1-Score': 0.0,
            'Positive_Samples': pos_samples
        })

path_b_results_df = pd.DataFrame(path_b_genre_f1_scores)

# Create scatter plot
fig, ax = plt.subplots(figsize=(14, 8))

# Path A scatter
scatter_a = ax.scatter(
    path_a_results_df['F1-Score'],
    path_a_results_df['Positive_Samples'],
    s=150,
    alpha=0.6,
    color='#2E86AB',
    edgecolor='black',
    linewidth=1.0,
    label='Path A (Audio→Genre)',
    zorder=2
)

# Path B scatter
scatter_b = ax.scatter(
    path_b_results_df['F1-Score'],
    path_b_results_df['Positive_Samples'],
    s=150,
    alpha=0.6,
    color='#A23B72',
    edgecolor='black',
    linewidth=1.0,
    label='Path B (Mood→Genre)',
    marker='^',
    zorder=2
)

# Customize plot
ax.set_xlabel('Genre F1-Score', fontsize=13, fontweight='bold')
ax.set_ylabel('Support (Number of Test Samples)', fontsize=13, fontweight='bold')
ax.set_title('Performance vs Support: Genre-Level F1 Scores\nPath A vs Path B', 
             fontsize=14, fontweight='bold', pad=20)

# Add grid
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Legend
ax.legend(fontsize=12, loc='upper left', framealpha=0.95)

# Add subtle background
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('white')

# Set y-axis to log scale for better visibility when there's large variance in support
ax.set_yscale('log')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "=" * 100)
print(" Performance vs support analysis")
print("=" * 100)

print("\n🔵 Path A (Audio→Genre) statistics:")
print("-" * 100)
print(f" Mean F1-Score: {path_a_results_df['F1-Score'].mean():.4f}")
print(f" Median F1-Score: {path_a_results_df['F1-Score'].median():.4f}")
print(f" F1-Score range: [{path_a_results_df['F1-Score'].min():.4f}, {path_a_results_df['F1-Score'].max():.4f}]")
print(f" Mean support: {path_a_results_df['Positive_Samples'].mean():.1f} samples")
print(f" Support range: [{path_a_results_df['Positive_Samples'].min()}, {path_a_results_df['Positive_Samples'].max()}]")

print("\n🟣 PATH B (Mood→Genre) STATISTICS:")
print("-" * 100)
print(f" Mean F1-Score: {path_b_results_df['F1-Score'].mean():.4f}")
print(f" Median F1-Score: {path_b_results_df['F1-Score'].median():.4f}")
print(f" F1-Score range: [{path_b_results_df['F1-Score'].min():.4f}, {path_b_results_df['F1-Score'].max():.4f}]")
print(f" Mean support: {path_b_results_df['Positive_Samples'].mean():.1f} samples")
print(f" Support range: [{path_b_results_df['Positive_Samples'].min()}, {path_b_results_df['Positive_Samples'].max()}]")

# Correlation between F1 and support for both paths
from scipy.stats import spearmanr, pearsonr

corr_a_pearson, pval_a_pearson = pearsonr(path_a_results_df['F1-Score'], path_a_results_df['Positive_Samples'])
corr_b_pearson, pval_b_pearson = pearsonr(path_b_results_df['F1-Score'], path_b_results_df['Positive_Samples'])

print("\n CORRELATION ANALYSIS (Pearson):")
print("-" * 100)
print(f" Path A (F1 vs Support): r = {corr_a_pearson:.4f}, p-value = {pval_a_pearson:.4f}")
print(f" Path B (F1 vs Support): r = {corr_b_pearson:.4f}, p-value = {pval_b_pearson:.4f}")

print("\n" + "=" * 100)
print(" Performance vs support scatter plot complete")
print("=" * 100)

In [ ]:
# Create figure with two subplots stacked vertically
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 18))

# --- Genre Bar Chart (vertical bars) ---
colors_genre = plt.cm.viridis(np.linspace(0.3, 0.9, len(df_genre_stats)))
bars1 = ax1.bar(df_genre_stats["tag"], df_genre_stats["tracks"],
                color=colors_genre, edgecolor="black", linewidth=0.7)

ax1.set_ylabel("Number of Track Appearances", fontsize=11, fontweight="bold")
ax1.set_xlabel("Genre Tag", fontsize=11, fontweight="bold")
ax1.set_title("Genre Tag Distribution\n(raw_30s_cleantags_50artists)",
              fontsize=13, fontweight="bold", pad=20)
ax1.grid(axis="y", alpha=0.3, linestyle="--")
ax1.tick_params(axis="x", labelrotation=90, labelsize=7)

# Add value labels above bars
for x, count in zip(df_genre_stats["tag"], df_genre_stats["tracks"]):
    ax1.text(x, count + 50, str(int(count)),
             ha="center", va="bottom", fontsize=7, fontweight="bold")

# --- Mood/Theme Bar Chart (vertical bars) ---
colors_mood = plt.cm.plasma(np.linspace(0.3, 0.9, len(df_mood_stats)))
bars2 = ax2.bar(df_mood_stats["tag"], df_mood_stats["tracks"],
                color=colors_mood, edgecolor="black", linewidth=0.7)

ax2.set_ylabel("Number of Track Appearances", fontsize=12, fontweight="bold")
ax2.set_xlabel("Mood/Theme Tag", fontsize=12, fontweight="bold")
ax2.set_title("Mood/Theme Tag Distribution\n(raw_30s_cleantags_50artists)",
              fontsize=14, fontweight="bold", pad=20)
ax2.grid(axis="y", alpha=0.3, linestyle="--")
ax2.tick_params(axis="x", labelrotation=90, labelsize=7)

# Add value labels above bars
for x, count in zip(df_mood_stats["tag"], df_mood_stats["tracks"]):
    ax2.text(x, count + 20, str(int(count)),
             ha="center", va="bottom", fontsize=9, fontweight="bold")

# Overall title
fig.suptitle("Audio Tag Frequency Analysis: 50-Artist Clean Tags (30s clips)",
             fontsize=16, fontweight="bold", y=0.995)

plt.tight_layout()
plt.show()
